# 17 - Manipulator Jacobian Control

## What / Why / How

**What we are trying to do:** Control an arm endpoint iteratively using the Jacobian and damped least squares.

**Why this matters:** Closed-form inverse kinematics is not always available. Jacobian methods are the practical backbone of many arm controllers.

**How we will do it:** Move a 2-link arm toward a target with resolved-rate IK, then inspect singular values to identify weak configurations.

## Goal

Move from inverse kinematics formulas to iterative Jacobian control.

You will implement:

- Resolved-rate IK
- Damped least squares
- Singularity awareness

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.control import resolved_rate_step

lengths = (1.0, 0.7)

def fk(q):
    q1, q2 = q
    l1, l2 = lengths
    elbow = np.array([l1*np.cos(q1), l1*np.sin(q1)])
    wrist = elbow + np.array([l2*np.cos(q1+q2), l2*np.sin(q1+q2)])
    return elbow, wrist

def jacobian(q):
    q1, q2 = q
    l1, l2 = lengths
    return np.array([
        [-l1*np.sin(q1)-l2*np.sin(q1+q2), -l2*np.sin(q1+q2)],
        [ l1*np.cos(q1)+l2*np.cos(q1+q2),  l2*np.cos(q1+q2)],
    ])

target = np.array([0.7, 1.0])
q = np.deg2rad([5.0, 20.0])
rows = []
for _ in range(120):
    elbow, wrist = fk(q)
    rows.append(np.r_[q, wrist])
    q = resolved_rate_step(q, target, fk, jacobian, gain=1.5, max_step=0.05, damping=1e-2)
rows = np.array(rows)
print("final endpoint:", rows[-1, 2:])
print("target error:", np.linalg.norm(rows[-1, 2:] - target))

In [ ]:
configs = [np.deg2rad([0, 0]), np.deg2rad([45, 90]), np.deg2rad([0, 180])]
for cfg in configs:
    singular_values = np.linalg.svd(jacobian(cfg), compute_uv=False)
    print("q deg", np.rad2deg(cfg), "singular values", singular_values)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    plt.plot(rows[:, 2], rows[:, 3], label="endpoint path")
    plt.scatter([target[0]], [target[1]], c="tab:red", label="target")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Try a target near the reach boundary.
2. Set damping very small near a singularity.
3. Add joint limits and reject steps outside them.